# Extract bounded article records from synthetic HTML and a sitemap

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Parse a sitemap, restrict accepted URLs to an expected origin, extract article paragraphs, and preserve provenance without fetching the network.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library and BeautifulSoup. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Author a tiny offline website

The reserved `example.org` domain is an identifier only. The tutorial never requests it. Each HTML string is written specifically for this example, including script/navigation text that should be removed.

In [ ]:
import xml.etree.ElementTree as ET
from hashlib import sha256
from urllib.parse import urlsplit, urlunsplit

from bs4 import BeautifulSoup

sitemap = """<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
<url><loc>https://example.org/docs/start</loc></url>
<url><loc>https://example.org/docs/reuse#overview</loc></url>
<url><loc>https://example.org/docs/start</loc></url>
</urlset>"""
pages = {
    "https://example.org/docs/start": "<html><nav>Menu</nav><main><article><h1>Getting started</h1><p>Measure the input before changing it.</p><script>ignore_this()</script><p>Record the output after each step.</p></article></main></html>",
    "https://example.org/docs/reuse": "<html><article><h1>Reuse a workflow</h1><p>Keep configuration separate from content.</p><p>Validate one small example first.</p></article></html>",
}

## 2. Parse only the expected sitemap dialect

For this bounded fixture, reject XML declarations that introduce entities or document types, then parse a namespaced URL set. Real remote sitemap ingestion needs byte limits at the HTTP layer too. URL validation here protects the downstream queue; it is not a complete network SSRF defense.

In [ ]:
def sitemap_urls(xml_text, allowed_host="example.org"):
    if len(xml_text.encode("utf-8")) > 100_000:
        raise ValueError("Sitemap exceeds example size limit")
    if "<!DOCTYPE" in xml_text.upper() or "<!ENTITY" in xml_text.upper():
        raise ValueError("Document types and entities are not supported")
    root = ET.fromstring(xml_text)
    namespace = "{http://www.sitemaps.org/schemas/sitemap/0.9}"
    if root.tag != namespace + "urlset":
        raise ValueError("Expected a sitemap URL set")
    result = []
    for node in root.findall(namespace + "url/" + namespace + "loc"):
        value = (node.text or "").strip()
        parts = urlsplit(value)
        if (
            parts.scheme != "https"
            or parts.hostname != allowed_host
            or parts.username
            or parts.password
            or parts.port not in (None, 443)
        ):
            raise ValueError("URL outside the allowed origin")
        canonical = urlunsplit((parts.scheme, parts.netloc, parts.path, parts.query, ""))
        if canonical not in result:
            result.append(canonical)
    return result


urls = sitemap_urls(sitemap)
assert urls == list(pages)
urls

## 3. Extract semantic content and attach stable hashes

Require an article and heading rather than accidentally indexing an error page or a navigation menu. Paragraph text is normalized by the parser. The content hash identifies the exact extracted text, not the complete original HTML.

In [ ]:
def article_record(url, html):
    if len(html.encode("utf-8")) > 100_000:
        raise ValueError("HTML exceeds example size limit")
    soup = BeautifulSoup(html, "html.parser")
    for element in soup(["script", "style", "nav"]):
        element.decompose()
    article = soup.find("article")
    if article is None or article.find("h1") is None:
        raise ValueError("Expected an article with a heading")
    title = article.find("h1").get_text(" ", strip=True)
    paragraphs = [p.get_text(" ", strip=True) for p in article.find_all("p")]
    paragraphs = [p for p in paragraphs if p]
    if not title or not paragraphs:
        raise ValueError("Article lacks meaningful content")
    text = "\n\n".join(paragraphs)
    return {
        "source_url": url,
        "title": title,
        "text": text,
        "content_sha256": sha256(text.encode("utf-8")).hexdigest(),
    }


records = [article_record(url, pages[url]) for url in urls]
assert len(records) == 2
assert "ignore_this" not in records[0]["text"]
assert "Menu" not in records[0]["text"]
records

## 4. Test absent content and unsafe URLs

The expected failure paths matter as much as the happy path: a parser that accepts a login page as an article creates incorrect search content without raising an error.

In [ ]:
for bad_html in ["<html>Unavailable</html>", "<article><h1>Empty</h1></article>"]:
    try:
        article_record(urls[0], bad_html)
    except ValueError:
        pass
    else:
        raise AssertionError("Nonarticle content accepted")
for bad_url in [
    "http://example.org/docs/start",
    "https://other.example/docs/start",
    "https://user@example.org/docs/start",
]:
    try:
        sitemap_urls(sitemap.replace(urls[0], bad_url))
    except ValueError:
        pass
    else:
        raise AssertionError("Disallowed origin accepted")
assert article_record(urls[0], pages[urls[0]]) == records[0]
print("Content, provenance and rejection checks passed.")

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

No crawler, embeddings or database is invoked. For a real source obtain permission, honor site policies, limit redirects and response sizes, validate every redirect destination, and add source-specific extraction tests. Rendering JavaScript and nested sitemap indexes are outside this example.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [BeautifulSoup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Python XML security considerations](https://docs.python.org/3/library/xml.html#xml-security)
- [Sitemap protocol](https://www.sitemaps.org/protocol.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.